# 한국 신분증 OCR 파인튜닝 - 빠른 시작

## 모델 선택 가이드
| 모델 | VRAM | 속도 | 정확도 | 추천 상황 |
|------|------|------|--------|----------|
| Qwen2.5-VL-2B + QLoRA | ≥12GB | 중간 | 높음 | GPU 서버 |
| Donut (200M) | ≥4GB | 빠름 | 중간 | 로컬/소형 GPU |

## 파이프라인
1. 합성 데이터 생성 → 2. 데이터셋 준비 → 3. 파인튜닝 → 4. 추론

In [ ]:
import subprocess, sys, os
os.chdir('..')  # 프로젝트 루트로

# GPU 확인
import torch
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')

## Step 1: 합성 데이터 생성

In [ ]:
# 운전면허증 100개 생성 (테스트용)
!python scripts/data_gen/generate_driver_license.py --output data/synthetic/driver_license --num 100

In [ ]:
# 여권 100개 생성
!python scripts/data_gen/generate_passport.py --output data/synthetic/passport --num 100

In [ ]:
# 생성된 이미지 확인
import matplotlib.pyplot as plt
from PIL import Image
import json
from pathlib import Path

# 운전면허증 샘플
dl_dir = Path('data/synthetic/driver_license')
dl_img = Image.open(list((dl_dir / 'images').glob('*.jpg'))[0])

# 여권 샘플
pp_dir = Path('data/synthetic/passport')
pp_img = Image.open(list((pp_dir / 'images').glob('*.jpg'))[0])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(dl_img)
axes[0].set_title('합성 운전면허증', fontsize=14)
axes[0].axis('off')
axes[1].imshow(pp_img)
axes[1].set_title('합성 여권', fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.show()

# 어노테이션 확인
with open(dl_dir / 'annotations.json') as f:
    anns = json.load(f)
print('운전면허증 어노테이션 예시:')
print(json.dumps(anns[0]['label'], ensure_ascii=False, indent=2))

## Step 2: 데이터셋 준비

In [ ]:
!python scripts/data_gen/prepare_dataset.py \
    --dl-dir data/synthetic/driver_license \
    --pp-dir data/synthetic/passport \
    --output data/processed

In [ ]:
from datasets import load_from_disk
dataset = load_from_disk('data/processed/ocr_dataset')
print(dataset)
print('\n학습 샘플 예시:')
sample = dataset['train'][0]
print(f'  doc_type: {sample["doc_type"]}')
print(f'  이미지 크기: {sample["image"].size}')
print(f'  응답: {sample["response"][:200]}...')

## Step 3: 파인튜닝 (Donut - 빠른 테스트)

In [ ]:
# Donut 파인튜닝 (GPU 메모리 적을 때)
!python scripts/train/train_donut.py \
    --dataset-path data/processed/ocr_dataset \
    --output-dir outputs/checkpoints/donut \
    --epochs 3 \
    --batch-size 2

In [ ]:
# Qwen2.5-VL QLoRA 파인튜닝 (VRAM ≥ 12GB)
# VRAM 확인 후 실행
import torch
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram_gb:.1f}GB')
    if vram_gb >= 12:
        print('→ Qwen2.5-VL QLoRA 파인튜닝 가능!')
        print('아래 셀을 실행하세요.')
    else:
        print('→ VRAM 부족. Donut 사용을 권장합니다.')

In [ ]:
# Qwen2.5-VL QLoRA (VRAM ≥ 12GB인 경우만 실행)
# !python scripts/train/train_qwen_lora.py \
#     --model_name_or_path Qwen/Qwen2.5-VL-2B-Instruct \
#     --dataset_path data/processed/ocr_dataset \
#     --use_4bit true \
#     --lora_r 16 \
#     --output_dir outputs/checkpoints/qwen_lora \
#     --num_train_epochs 3 \
#     --per_device_train_batch_size 1 \
#     --gradient_accumulation_steps 8 \
#     --learning_rate 2e-4 \
#     --bf16 true

## Step 4: 추론 테스트

In [ ]:
import sys
sys.path.insert(0, 'scripts/inference')
from predict import DonutOCRPredictor
from PIL import Image
import json

# Donut 추론
predictor = DonutOCRPredictor('outputs/checkpoints/donut/final')

# 테스트 이미지
test_img = list(Path('data/synthetic/driver_license/images').glob('*.jpg'))[0]
result = predictor.predict(test_img, doc_type='driver_license')
print('추출 결과:')
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Qwen2.5-VL 추론 (파인튜닝 없이 기본 모델로도 테스트 가능)
# from predict import QwenOCRPredictor
# predictor = QwenOCRPredictor()
# result = predictor.predict(test_img, doc_type='driver_license')
# print(json.dumps(result, ensure_ascii=False, indent=2))